## Load the File

In [5]:
from pathlib import Path
from bs4 import BeautifulSoup

# Finds the file by pattern instead of hardcoding the date
path = next(Path("../data/raw").glob("AAPL_10-K_*.html"))
html = path.read_text(encoding="utf-8")
# Turns the HTML string into a navigable tree of tags
soup = BeautifulSoup(html, "lxml")

print(path.name, f"{len(html):,} characters")

/var/folders/g0/83td4ygx4q3704543wrw62180000gn/T/ipykernel_68211/2262744922.py:8: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


AAPL_10-K_2026-11-03.html 1,650,108 characters


## Remove Hidden XBRL Data

In [6]:
# Finds every tag styled display:none
hidden = soup.find_all(style=lambda s: s and "display:none" in s.replace(" ", ""))
print(f"Hidden elements: {len(hidden)}")

for tag in hidden:
	tag.decompose()   # Removes the tag from the document entirely

Hidden elements: 77


## Extract Text

In [7]:
# Strips all tags nd keeps only the text
text = soup.get_text(separator="\n")
# Trims each line and drops empty ones
lines = [line.strip() for line in text.splitlines() if line.strip()]

print(f"{len(lines):,} lines")
print("\n".join(lines[:40]))

4,162 lines
aapl-20260926
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM
10-K
(Mark One)
☒
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended
September 26
, 2026
or
☐
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from
to
.
Commission File Number:
001-36743
Apple Inc.
(Exact name of Registrant as specified in its charter)
California
94-2404110
(State or other jurisdiction
of incorporation or organization)
(I.R.S. Employer Identification No.)
One Apple Park Way
MS 927-4GC
Cupertino
,
California
95014
(Address of principal executive offices)
(Zip Code)
(
408
)
996-1010
(Registrant’s telephone number, including area code)


## How Do Tables Look as Plain Text?

In [8]:
# Collects every line number containing "Total net sales"
hits = [i for i, line in enumerate(lines) if "Total net sales" in line]
print(f"{len(hits)} matches")

i = hits[0]
for line in lines[i - 5 : i + 10]:
	print(repr(line))

5 matches
'%'
'33,696'
'10'
'%'
'30,658'
'Total net sales'
'$'
'478,834'
'15'
'%'
'$'
'416,161'
'6'
'%'
'$'


## Extract a Table as Rows

In [9]:
tables = soup.find_all("table")
print(f"{len(tables)} tables")

# Picks the first table containing "Total net sales"
target = next(t for t in tables if "Total net sales" in t.get_text())
for tr in target.find_all("tr"):   # Iterates over the table's rows
	cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
	print(cells)

68 tables
['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']
['', '2026', '', 'Change', '', '2025', '', 'Change', '', '2024']
['Americas', '$', '198,083', '', '', '11', '%', '', '$', '178,353', '', '', '7', '%', '', '$', '167,045', '']
['Europe', '127,604', '', '', '15', '%', '', '111,032', '', '', '10', '%', '', '101,328', '']
['Greater China', '82,378', '', '', '28', '%', '', '64,377', '', '', '(4)', '%', '', '66,952', '']
['Japan', '30,908', '', '', '8', '%', '', '28,703', '', '', '15', '%', '', '25,052', '']
['Rest of Asia Pacific', '39,861', '', '', '18', '%', '', '33,696', '', '', '10', '%', '', '30,658', '']
['Total net sales', '$', '478,834', '', '', '15', '%', '', '$', '416,161', '', '', '6', '%', '', '$', '391,035', '']


## Clean a Row

In [10]:
import re

def clean_row(cells):
	out = []
	for c in cells:
		# Removes whitespace right after (. \s+ means one or more whitespace characters.
		# re is Python's regular expression library.
		c = re.sub(r"\(\s+", "(", c.strip())
		# Remove whitespace right before ), turning ( 321 ) into (321)
		c = re.sub(r"\s+\)", ")", c)

		# Skips empty cells and lone $ signs, which carry little meaning and only
		# misalign columns
		if c in ("", "$"):
			continue
		# Attaches lone % or ) to the previous cell
		if c in ("%", ")", ")%") and out:
			out[-1] += c
			continue
		out.append(c)
	return out

## Convert to Markdown

In [11]:
def is_section(row):
	return len(row) == 1 and row[0].endswith(":")

# A pattern matching amounts and percentages like 6,617, 39.3%, (321), (4)%
NUMBER = re.compile(r"^\(?-?[\d,]*\.?\d+\)?%?$")
# Matches four-digit years from 1900 to 2099
YEAR = re.compile(r"^(19|20)\d{2}$")

# Return False if any value cell (excluding the label) is a number other than a year
def looks_like_header(row):
	return not any(NUMBER.match(c) and not YEAR.match(c) for c in row[1:])

def table_to_markdown(table):
	rows = []
	for tr in table.find_all("tr"):
		cells = [td.get_text(" ", strip=True) for td in tr.find_all(["td", "th"])]
		row = clean_row(cells)
		if row:   # Drops rows that are empty after cleaning, like the spacer row
			rows.append(row)

	if not rows:
		return ""

	width = max(len(r) for r in rows)
	n_cols = width - 1

	# 1. Header: leading rows without a label cell (shorter than data rows)
	header = [""] * n_cols
	corner = ""
	# Takes rows from the top as headers while they're shorter than data rows, stopping at the first section title
	while rows and len(rows[0]) < width and not is_section(rows[0]):
		hr = rows.pop(0)
		# A single-cell header like "Years Ended" spans all columns, so it's copied to each one
		if len(hr) == 1:
			hr = hr * n_cols
		# Pads short header rows on the left ot right-align them, as before
		hr = [""] * (n_cols - len(hr)) + hr
		# Joins header rows column by column
		header = [f"{h} {x}".strip() for h, x in zip(header, hr)]

	# 1b. Fallback: a full-width first row with no amounts is a header
	if (
		not any(header) 
		and len(rows) > 1 
		and not is_section(rows[0])   # never treat a section title as a header
		and looks_like_header(rows[0])
	):
		hr = rows.pop(0)
		corner = hr[0]
		header = hr[1:] + [""] * (n_cols - len(hr[1:]))

	# 2. Body: prefix each row with its section title
	body = []
	section = ""
	for r in rows:
		if is_section(r):
			# Remembers the section title and skips the row itself, since its meaning
			# moves into the labels below
			section = r[0].rstrip(":")
			continue
		label = r[0]
		if section and not label.startswith("Total"):
			label = f"{section} - {label}"
		body.append([label] + r[1:] + [""] * (width - len(r)))
		# A "Total" row closes the section, so the next row doesn't inherit it
		if r[0].startswith("Total"):
			section = ""

	# 3. Markdown
	lines = [f"| {corner} | " + " | ".join(header) + " |", "|" + " --- |" * width]
	lines += ["| " + " | ".join(r) + " |" for r in body]
	return "\n".join(lines)

print(table_to_markdown(target))

|  | 2026 | Change | 2025 | Change | 2024 |
| --- | --- | --- | --- | --- | --- |
| Americas | 198,083 | 11% | 178,353 | 7% | 167,045 |
| Europe | 127,604 | 15% | 111,032 | 10% | 101,328 |
| Greater China | 82,378 | 28% | 64,377 | (4)% | 66,952 |
| Japan | 30,908 | 8% | 28,703 | 15% | 25,052 |
| Rest of Asia Pacific | 39,861 | 18% | 33,696 | 10% | 30,658 |
| Total net sales | 478,834 | 15% | 416,161 | 6% | 391,035 |


In [12]:
candidates = [t for t in tables if "Total net sales" in t.get_text()]
print(f"{len(candidates)} candidate tables\n")

for i, t in enumerate(candidates):
	print(f"===== Table {i} =====")
	print(table_to_markdown(t))
	print()

5 candidate tables

===== Table 0 =====
|  | 2026 | Change | 2025 | Change | 2024 |
| --- | --- | --- | --- | --- | --- |
| Americas | 198,083 | 11% | 178,353 | 7% | 167,045 |
| Europe | 127,604 | 15% | 111,032 | 10% | 101,328 |
| Greater China | 82,378 | 28% | 64,377 | (4)% | 66,952 |
| Japan | 30,908 | 8% | 28,703 | 15% | 25,052 |
| Rest of Asia Pacific | 39,861 | 18% | 33,696 | 10% | 30,658 |
| Total net sales | 478,834 | 15% | 416,161 | 6% | 391,035 |

===== Table 1 =====
|  | 2026 | Change | 2025 | Change | 2024 |
| --- | --- | --- | --- | --- | --- |
| iPhone | 254,098 | 21% | 209,586 | 4% | 201,183 |
| Mac | 37,929 | 13% | 33,708 | 12% | 29,984 |
| iPad | 27,929 | —% | 28,023 | 5% | 26,694 |
| Wearables, Home and Accessories | 35,577 | —% | 35,686 | (4)% | 37,005 |
| Services (1) | 123,301 | 13% | 109,158 | 14% | 96,169 |
| Total net sales | 478,834 | 15% | 416,161 | 6% | 391,035 |

===== Table 2 =====
|  | Years Ended September 26, 2026 | Years Ended September 27, 2025 | Years 

In [13]:
failed = []
for i, t in enumerate(tables):
	try:
		table_to_markdown(t)
	except Exception as e:
		failed.append((i, type(e).__name__, str(e)))

print(f"{len(tables) - len(failed)}/{len(tables)} tables converted")
for f in failed:
	print(f)

68/68 tables converted


In [14]:
for i, _, _ in failed:
    text = tables[i].get_text(" ", strip=True)
    print(i, repr(text[:100]))

In [15]:
# Converts all 68 tables once and keeps the results for the checks below
results = [table_to_markdown(t) for t in tables]

empty = [i for i, md in enumerate(results) if md == ""]
no_header = [
	i for i, md in enumerate(results)
    if md and not re.search(r"\w", md.splitlines()[0])
]

print(f"Empty: {len(empty)}")
print(f"Converted: {len(results) - len(empty)}")
print(f"No header found: {len(no_header)} -> {no_header}")

Empty: 8
Converted: 60
No header found: 5 -> [17, 34, 49, 53, 60]


In [16]:
for i in no_header:
	lines = results[i].splitlines()
	print(f"===== Table {i} ({len(lines) - 2} rows) =====")
	print("\n".join(lines[:5]))
	print()

===== Table 17 (3 rows) =====
|  |  |  |  |
| --- | --- | --- | --- |
| Gross margin percentage - Products | 39.3% | 36.8% | 37.2% |
| Gross margin percentage - Services | 76.2% | 75.4% | 73.9% |
| Total gross margin percentage | 48.8% | 46.9% | 46.2% |

===== Table 34 (7 rows) =====
|  |  |
| --- | --- |
| 2027 | 6,617 |
| 2028 | 6,044 |
| 2029 | 5,517 |

===== Table 49 (7 rows) =====
|  |  |
| --- | --- |
| 2027 | 9,991 |
| 2028 | 9,300 |
| 2029 | 5,175 |

===== Table 53 (7 rows) =====
|  |  |
| --- | --- |
| 2027 | 6,712 |
| 2028 | 6,418 |
| 2029 | 5,515 |

===== Table 60 (1 rows) =====
|  |  |
| --- | --- |
| How We Addressed the Matter in Our Audit | We tested controls relating to the evaluation of uncertain tax positions, including controls over management’s assessment as to whether tax positions are more likely than not to be sustained, management’s process to measure the benefit of its tax positions that qualify for recognition, and the related disclosures. We evaluated the Com

## Parse the Whole Document

In [21]:
TABLE_MARKER = re.compile(r"\[\[TABLE_(\d+)\]\]")
# Block-level tags; line breaks are inserted only at their ends
BLOCK_TAGS = ["div", "p", "br", "li", "h1", "h2", "h3", "h4"]

# context_lines: how many preceding lines to attach as context; a tunable parameter
def parse_10k(html, context_lines=2):
	soup = BeautifulSoup(html, "lxml")

	# 1. Remove hidden XBRL data
	for tag in soup.find_all(style=lambda s: s and "display:none" in s.replace(" ", "")):
		tag.decompose()

	# 2. Convert tables to markdown and leave a marker in their place
	tables_md = []
	for table in soup.find_all("table"):
		md = table_to_markdown(table)
		if md:
			# Replaces the table with a marker on its own line
			table.replace_with(f"\n[[TABLE_{len(tables_md)}]]\n")
			tables_md.append(md)
		else:
			table.decompose()

	# 3. Break lines only at the end of block-level tags
	for tag in soup.find_all(BLOCK_TAGS):
		tag.append("\n")
	body = soup.body or soup   # skip <head>
	lines = [re.sub(r"\s+", " ", line).strip() for line in body.get_text().splitlines()]
	lines = [line for line in lines if line]   # drop empty lines


	# 4. Walk lines in order, building text and table blocks
	blocks = []
	buffer = []   # Collects text lines until a table appears, then becomes one text block
	recent = []   # All text lines seen so far. Unlike buffer, it's never cleared
	for line in lines:
		m = TABLE_MARKER.fullmatch(line)   # Checks that the entire line is a marker
		if m:
			if buffer:
				blocks.append({"type": "text", "content": "\n".join(buffer)})
				buffer = []
			blocks.append({
				"type": "table",
				# Joins the last two text lines before the table as its context
				"context": " ".join(recent[-context_lines:]),
				# Extracts the captured number
				"content": tables_md[int(m.group(1))],
			})
		else:
			buffer.append(line)
			recent.append(line)

	if buffer:
		blocks.append({"type": "text", "content": "\n".join(buffer)})
	return blocks

## Check the Result

In [19]:
blocks = parse_10k(html)

n_tables = sum(b["type"] == "table" for b in blocks)   # Counts the table blocks
print(f"{len(blocks)} blocks ({n_tables} tables, {len(blocks) - n_tables} text)\n")

for keyword in ["Gross margin percentage", "Total net sales"]:
	for b in blocks:
		if b["type"] == "table" and keyword in b["content"]:
			print("CONTEXT:", b["context"])
			print(b["content"])
			print()
			break

117 blocks (60 tables, 57 text)

CONTEXT: Gross Margin Products and Services gross margin and gross margin percentage for 2026, 2025 and 2024 were as follows (dollars in millions):
|  |  |  |  |
| --- | --- | --- | --- |
| Gross margin percentage - Products | 39.3% | 36.8% | 37.2% |
| Gross margin percentage - Services | 76.2% | 75.4% | 73.9% |
| Total gross margin percentage | 48.8% | 46.9% | 46.2% |

CONTEXT: Segment Operating Performance The following table shows net sales by reportable segment for 2026, 2025 and 2024 (dollars in millions):
|  | 2026 | Change | 2025 | Change | 2024 |
| --- | --- | --- | --- | --- | --- |
| Americas | 198,083 | 11% | 178,353 | 7% | 167,045 |
| Europe | 127,604 | 15% | 111,032 | 10% | 101,328 |
| Greater China | 82,378 | 28% | 64,377 | (4)% | 66,952 |
| Japan | 30,908 | 8% | 28,703 | 15% | 25,052 |
| Rest of Asia Pacific | 39,861 | 18% | 33,696 | 10% | 30,658 |
| Total net sales | 478,834 | 15% | 416,161 | 6% | 391,035 |



/var/folders/g0/83td4ygx4q3704543wrw62180000gn/T/ipykernel_68211/999457573.py:7: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


In [20]:
for b in blocks[:6]:
	if b["type"] == "text":
		print(b["content"][:400])
		print("-----")

aapl-20260926
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
FORM 10-K
(Mark One)
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended September 26, 2026
or
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from              to             .
Commis
-----
(408) 996-1010
(Registrant’s telephone number, including area code)
Securities registered pursuant to Section 12(b) of the Act:
-----
Securities registered pursuant to Section 12(g) of the Act:  None
Indicate by check mark if the Registrant is a well-known seasoned issuer, as defined in Rule 405 of the Securities Act.
Yes  ☒     No  ☐
Indicate by check mark if the Registrant is not required to file reports pursuant to Section 13 or Section 15(d) of the Act.
Yes  ☐     No  ☒
Indicate by check mark whether the Registrant (1) has f
-----


In [22]:
from collections import Counter

blocks = parse_10k(html)
text_lines = [
	line
	for b in blocks if b["type"] == "text"
	for line in b["content"].splitlines()
]

for line, n in Counter(text_lines).most_common(10):
	print(n, repr(line[:80]))

/var/folders/g0/83td4ygx4q3704543wrw62180000gn/T/ipykernel_68211/413757211.py:7: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


8 'Apple Inc.'
5 'See accompanying Notes to Consolidated Financial Statements.'
4 'The information required by this Item will be included in the 2027 Proxy Stateme'
3 'Yes ☒ No ☐'
3 'None.'
2 'Yes ☐ No ☒'
2 'iPhone'
2 'Mac'
2 'iPad'
2 'Wearables, Home and Accessories'


In [23]:
page_nums = [line for line in text_lines if re.fullmatch(r"\d{1,3}", line)]
print(len(page_nums), page_nums[:15])

0 []
